In [0]:
%run ../notebooks/create_secret


In [0]:
%pip install pymongo
%pip install pyyaml

dbutils.library.restartPython()

In [0]:
import datetime
import json
import os
import time 
import bson
from pymongo import MongoClient
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType
 
class MongoExtractor:
    def __init__(self, database: str = "sample_mflix"):
        self.database = database
        self.uri = dbutils.secrets.get(scope="conn-db", key="cnn-mongodb-sampleflix")
        self.client = MongoClient(self.uri)

    def extract(self, collection, modo_carga="full", campo_watermark=None, watermark_value=None):
        data = self.client[self.database][collection]

        filtro = {}
        if modo_carga == "incremental" and campo_watermark and watermark_value is not None:
            filtro = {campo_watermark: {"$gt": watermark_value}}

        cursor = data.find(filtro)
        return list(cursor)  # simplificado por enquanto — no R2 será feita a implementação da leitura paginada

In [0]:
import json
import yaml

from datetime import date

class LandingWriter:
    def __init__(self, dbutils, base_path="/Volumes/mongo_db/bronze/sample_mflix"):
        self.dbutils = dbutils
        self.base_path = base_path

    def write(self, dados, collection):
        if not dados:
            return None

        hoje = date.today().isoformat()
        path = f"{self.base_path}/{collection}/_ingestion_date={hoje}"
        self.dbutils.fs.mkdirs(path)

        file_path = f"{path}/{collection}.json"
        # cada documento em uma linha (formato JSON Lines)
        conteudo = "\n".join(json.dumps(doc, default=str) for doc in dados)
        self.dbutils.fs.put(file_path, conteudo, overwrite=True)

        return file_path

In [0]:
mongo_db = MongoExtractor()
writer = LandingWriter(dbutils)

CONFIG_PATH = "../config/collections.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print(config)

In [0]:
def run(config):
    database = config["database"]

    for item in config["collections"]:
        dados = mongo_db.extract(
            #database=database,
            collection=item["collection"],
            #modo_carga=item["modo_carga"],
            #campo_watermark=item["campo_watermark"],
        )

        writer.write(dados, item["collection"])  # <-- novo passo



In [0]:
d = run(config)

In [0]:
df = (
    spark.read
    .format("json")
    .load("/Volumes/mongo_db/bronze/sample_mflix/movies/_ingestion_date=2026-08-25/movies.json")
    )
#df.printSchema()
df.count()